In [1]:
import os
import torch
import pandas as pd
import scanpy as sc
import SpatialGlue
from SpatialGlue.preprocess import clr_normalize_each_cell, pca
from SpatialGlue.preprocess import fix_seed
from SpatialGlue.preprocess import construct_neighbor_graph
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue
from SpatialGlue.utils import clustering
import matplotlib.pyplot as plt
from pandas import ExcelWriter

In [ ]:
sample='sample_tmp'
binsize=100

input_path=f'/data/work/cite_seq/result/{sample}/bin{binsize}'
RNA_path=f'{input_path}/{sample}.bin{binsize}.RNA.h5ad'
protein_path=f'{input_path}/{sample}.bin{binsize}.protein.h5ad'

In [4]:
adata_RNA = sc.read_h5ad(RNA_path)
adata_protein=sc.read_h5ad(protein_path)

In [5]:
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
data_type = 'Stereo-CITE-seq'
random_seed = 2022
fix_seed(random_seed)

In [ ]:
data = construct_neighbor_graph(adata_RNA, adata_protein, datatype=data_type)
# define model
model = Train_SpatialGlue(data, datatype=data_type, device=device)
# train model
output = model.train()

In [ ]:
adata = adata_RNA.copy()
adata.obsm['emb_latent_omics_RNA'] = output['emb_latent_omics1']
adata.obsm['emb_latent_omics_protein'] = output['emb_latent_omics2']
adata.obsm['SpatialGlue'] = output['SpatialGlue']
adata.obsm['alpha'] = output['alpha']
adata.obsm['alpha_omics_RNA'] = output['alpha_omics1']
adata.obsm['alpha_omics_protein'] = output['alpha_omics2']

adata.write_h5ad(os.path.join(input_path, f"{sample}.after_model_train.RNA.h5ad"))

In [ ]:
adata_path=f'{input_path}/{sample}.after_model_train.RNA.h5ad'
print(adata_path)
adata=sc.read_h5ad(adata_path)

In [ ]:
tool = 'mclust'
for cluster_num in range(2,24):
    cluster_name=f'SpatialGlue_cluster{cluster_num}'
    cluster_path=f'{input_path}/cluster{cluster_num}'
    os.makedirs(cluster_path, exist_ok=True)
    print(cluster_path)
    
    clustering(adata, key='SpatialGlue', add_key=cluster_name, n_clusters=cluster_num, method=tool, use_pca=True)
    sc.pp.neighbors(adata, use_rep='SpatialGlue', n_neighbors=30)
    sc.tl.umap(adata, min_dist=0.5)
    
    output_name = f"{sample}_{binsize}_cluster_{cluster_num}"
    fig, ax_list = plt.subplots(1, 2, figsize=(7, 3))
    sc.pl.umap(adata, color=cluster_name, ax=ax_list[0], title=f'{sample}_umap', s=2, show=False)
    sc.pl.embedding(adata, basis='spatial', color=cluster_name, ax=ax_list[1], title=f'{sample}_embedding', s=1, show=False)
    ax_list[1].invert_yaxis()
    plt.tight_layout(w_pad=0.3)
    plt.savefig(f'{cluster_path}/{output_name}.png', dpi=600, bbox_inches='tight')
    plt.close()
    
    
    DEG_path=f'{cluster_path}/diff_exp_gene'
    os.makedirs(DEG_path, exist_ok=True)
    print(DEG_path)
    adata.obs[cluster_name] = adata.obs[cluster_name].astype(str).astype('category')
    sc.tl.rank_genes_groups(adata, cluster_name, method="wilcoxon", use_raw=False)
    groups=list(adata.obs[cluster_name].unique())
    dfs = []
    for group in groups:
        df = sc.get.rank_genes_groups_df(adata, group=group)
        df['cluster'] = group
        dfs.append(df)
    df_all = pd.concat(dfs)
    df_all = df_all.sort_values(by="logfoldchanges", ascending=False)
    df_all.to_csv(f'{DEG_path}/{sample}_cluster{cluster_num}.csv',index=False)
    print(f'{DEG_path}/{sample}_cluster{cluster_num}.csv')
    
    filtered_DEG=df_all.loc[(df_all['pvals_adj']<0.05) & (df_all['logfoldchanges']>0.25)]
    filtered_DEG = filtered_DEG.sort_values(by='logfoldchanges', ascending=False)
    zhang_M_anno=pd.read_csv('/data/work/file/zhang_M.csv')
    zhang_M_anno.rename(columns={'ID':'names'},inplace=True)
    genes_zhang_M_anno=pd.merge(filtered_DEG,zhang_M_anno,on='names',how='left')

    cluster_list=list(genes_zhang_M_anno.loc[:,'cluster'].unique())
    
    output_path = f'{DEG_path}/{sample}_cluster{cluster_num}_anno.xlsx'
    with ExcelWriter(output_path) as writer:
        for cluster in cluster_list:
            cluster_data = genes_zhang_M_anno[genes_zhang_M_anno['cluster'] == cluster]
            sheet_name = f'cluster{cluster}'
            cluster_data.to_excel(writer, sheet_name=sheet_name, index=False)
    
    DEG_cluster_path=f'{DEG_path}/{sample}_cluster{cluster_num}_anno'
    os.makedirs(DEG_cluster_path, exist_ok=True)
    print(DEG_cluster_path)
    for cluster in cluster_list:
        cluster_csv=genes_zhang_M_anno[genes_zhang_M_anno['cluster']==cluster]
        cluster_csv = cluster_csv.sort_values(by="logfoldchanges", ascending=False)
        cluster_csv.to_csv(f'{DEG_cluster_path}/{cluster}.csv',index=False)
        
adata.write_h5ad(os.path.join(input_path, f"{sample}.after_cluster.RNA.h5ad"))
print(f"Saved h5ad file: {input_path}/{sample}.after_cluster.RNA.h5ad")

In [ ]:
for cluster_num in range(2,24):
    cluster_name=f'SpatialGlue_cluster{cluster_num}'
    cluster_path=f'{input_path}/cluster{cluster_num}'
    DEP_path=f'{cluster_path}/diff_exp_protein'
    os.makedirs(DEP_path, exist_ok=True)
    print(DEP_path)
    
    adata_protein = adata_protein[adata_protein.obs_names.isin(adata.obs_names)]
    adata_protein.obs = adata_protein.obs.join(adata.obs[[cluster_name]])
    adata_protein.obs['n_proteins'] = (adata_protein.X > 0).sum(axis=1)
    
    sc.tl.rank_genes_groups(adata_protein, cluster_name, method="wilcoxon", use_raw=False)
    groups = list(adata_protein.obs[cluster_name].unique())
    dfs = []
    for group in groups:
        df = sc.get.rank_genes_groups_df(adata_protein, group=group)
        df['cluster'] = group
        dfs.append(df)
    df_all = pd.concat(dfs)
    df_all = df_all.sort_values(by="logfoldchanges", ascending=False)
    df_all.to_csv(f'{DEP_path}/{sample}_cluster{cluster_num}.csv',index=False)
    print(f'{DEG_path}/{sample}_cluster{cluster_num}.csv')
    
    filtered_DEG=df_all.loc[(df_all['pvals_adj']<0.05)]
    protein_anno=pd.read_csv('/data/work/file/protein_anno.csv')
    protein_anno.rename(columns={'PIDName':'names'},inplace=True)
    protein_after_anno=pd.merge(filtered_DEG,protein_anno,on='names',how='left')
    cluster_list=list(df_all.loc[:,'cluster'].unique())
    
    output_path = f'{DEP_path}/{sample}_cluster{cluster_num}_protein_anno.xlsx'
    with ExcelWriter(output_path) as writer:
        for cluster in cluster_list:
            cluster_data = protein_after_anno[protein_after_anno['cluster'] == cluster]
            sheet_name = f'cluster{cluster}'
            cluster_data.to_excel(writer, sheet_name=sheet_name, index=False)
    
    DEP_cluster_path=f'{DEP_path}/{sample}_cluster{cluster_num}_protein_anno'
    os.makedirs(DEP_cluster_path, exist_ok=True)
    print(DEP_cluster_path)
    for cluster in cluster_list:
        cluster_csv=protein_after_anno[protein_after_anno['cluster']==cluster]
        cluster_csv = cluster_csv.sort_values(by="logfoldchanges", ascending=False)
        cluster_csv.to_csv(f'{DEP_cluster_path}/{cluster}.csv',index=False)
        
adata_protein.write_h5ad(os.path.join(input_path, f"{sample}.after_cluster.protein.h5ad"))
print(f"Saved h5ad file: {input_path}/{sample}.after_cluster.protein.h5ad")

In [ ]:
print('gugugu')